# 01 - Ingestion

Scope: ingest 5 MOM.gov.sg pages across 5 categories (salary, working hours, work
permit conditions, medical insurance, help) as the corpus. Output is a structured,
cached record per page that `02_chunking` consumes — this notebook does not chunk,
embed, or index.

## Step 1: Setup

Imports, and the source URL list as an explicit constant (not hardcoded inline later).

In [2]:
from pathlib import Path

import requests
import trafilatura

# category is assigned per-URL (not a blanket constant) since the corpus now spans
# multiple real categories.
# Note: work-injury (WICA) and housing were dropped -- both are landing/hub pages
# with ~90% navigation and no substantive content of their own (confirmed by
# fetching them directly); the real content lives on their sub-pages, not yet
# sourced. See CLAUDE.md Domain notes for deferred categories.
SOURCE_URLS = [
    {
        "url": "https://www.mom.gov.sg/employment-practices/salary/paying-salary",
        "category": "salary",
    },
    {
        "url": "https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days",
        "category": "working-hours",
    },
    {
        "url": "https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/work-permit-conditions",
        "category": "work-permit",
    },
    {
        "url": "https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/medical-insurance",
        "category": "medical",
    },
    {
        "url": "https://www.mom.gov.sg/contact-us",
        "category": "help",
    },
]

# Constant for now -- all sources are MOM guidance pages -- kept as a separate field
# for when other source types (FAQs, forms, third-party guidance) are added later.
CONTENT_TYPE = "official_guidance"

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

## Step 2: Fetch raw HTML

Request each URL (explicit User-Agent, timeout, raise/log on non-200).
Only 2 pages, so no concurrency/rate-limiting machinery needed here.

In [3]:
raw_html = {}

for source in SOURCE_URLS:
    url = source["url"]
    response = requests.get(
        url,
        headers={"User-Agent": "migrantBuddy-ingestion/0.1 (research prototype)"},
        timeout=15,
    )
    response.raise_for_status()
    raw_html[url] = response.text

{url: len(html) for url, html in raw_html.items()}

{'https://www.mom.gov.sg/employment-practices/salary/paying-salary': 62344,
 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days': 67335,
 'https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/work-permit-conditions': 71915,
 'https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/medical-insurance': 67201,
 'https://www.mom.gov.sg/contact-us': 52276}

## Step 3: Cache raw HTML to disk

Save untouched HTML to `data/raw/` keyed by a slug of the URL, before any
parsing. Keeps ingestion reproducible/offline-repeatable and gives a debugging
reference if extraction looks wrong later — re-run parsing without re-fetching.

In [4]:
import re

RAW_DIR.mkdir(parents=True, exist_ok=True)


def slugify(url: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "-", url).strip("-")


for url, html in raw_html.items():
    (RAW_DIR / f"{slugify(url)}.html").write_text(html, encoding="utf-8")

sorted(p.name for p in RAW_DIR.glob("*.html"))

['https-www-mom-gov-sg-contact-us.html',
 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days.html',
 'https-www-mom-gov-sg-employment-practices-salary-paying-salary.html',
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-housing.html',
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-medical-insurance.html',
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions.html',
 'https-www-mom-gov-sg-workplace-safety-and-health-work-injury-compensation.html']

## Step 4: Extract main content text

Use trafilatura to strip nav/footer/boilerplate and keep the substantive page text.

In [5]:
extracted_text = {}

for url, html in raw_html.items():
    extracted_text[url] = trafilatura.extract(
        html,
        output_format="markdown",
        include_tables=True,
        include_comments=False,
        favor_precision=False,
    )

{url: len(text or "") for url, text in extracted_text.items()}

{'https://www.mom.gov.sg/employment-practices/salary/paying-salary': 3940,
 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days': 6867,
 'https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/work-permit-conditions': 3053,
 'https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/medical-insurance': 2917,
 'https://www.mom.gov.sg/contact-us': 1004}

## Step 5: Assemble structured record per page

One record per page: `{document_id, url, title, authority, category, content_type,
language, source_type, fetched_at, text}`. `document_id` is derived from the URL (via
`slugify`, Step 3) so it's stable and automatic rather than hand-typed. `category`
comes from `SOURCE_URLS` (per-URL, not a blanket constant). Tables are already inline
in `text` as markdown (Step 4), correctly positioned relative to their surrounding
headings — no separate `tables` field needed. This is the contract `02_chunking` will
consume, so its shape matters more than any single field's content.

In [6]:
from datetime import datetime, timezone

records = []

for source in SOURCE_URLS:
    url = source["url"]
    metadata = trafilatura.extract_metadata(raw_html[url])
    records.append(
        {
            "document_id": slugify(url),
            "url": url,
            "title": metadata.title if metadata else None,
            "authority": "MOM",
            "category": source["category"],
            "content_type": CONTENT_TYPE,
            "language": "en",
            "source_type": "html",
            "fetched_at": datetime.now(timezone.utc).isoformat(),
            "text": extracted_text[url],
        }
    )

records[0]

{'document_id': 'https-www-mom-gov-sg-employment-practices-salary-paying-salary',
 'url': 'https://www.mom.gov.sg/employment-practices/salary/paying-salary',
 'title': 'Paying salary',
 'authority': 'MOM',
 'category': 'salary',
 'content_type': 'official_guidance',
 'language': 'en',
 'source_type': 'html',
 'fetched_at': '2026-07-21T07:16:33.188462+00:00',
 'text': 'In accordance with the Employment Act, your employer must pay your salary at least once a month and within 7 days after the end of the salary period. There are exceptions for overtime, resignation without notice and other situations.\n\n     \n \n            \n     If you are facing a salary dispute, you should get advice from the Tripartite Alliance for Dispute Management (TADM). \n\nUse the "Ask TADM" chatbot for immediate answers or make an appointment to speak with a TADM advisory officer. You can also file your claims through the \n\nTADM eServices, or approach your union if you are a union member. \n\n## What is sal

## Step 6: Save parsed output

Write the assembled records to `data/processed/ingested.json` (gitignored) as the
durable output of this notebook.

In [7]:
import json

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "ingested.json"
output_path.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")

output_path

WindowsPath('../data/processed/ingested.json')

## Step 7: Sanity checks

Confirm both pages produced non-trivial text, expected tables were found, and nothing
silently failed (e.g. a page reduced to near-empty text after boilerplate stripping).

In [8]:
MIN_TEXT_LENGTH = 500

for record in records:
    assert record["title"], f"Missing title: {record['url']}"
    assert record["text"], f"Empty text: {record['url']}"
    assert len(record["text"]) >= MIN_TEXT_LENGTH, (
        f"Suspiciously short text ({len(record['text'])} chars): {record['url']}"
    )
    # Warning, not a hard assert: some legitimate pages (e.g. a short contact page)
    # have real content without markdown heading structure -- confirmed by manual
    # review, not a sign of broken extraction. The length check above is what
    # actually catches genuinely empty/broken pages.
    if "#" not in record["text"]:
        print(f"WARNING: no heading structure found (may be legitimate): {record['url']}")

for record in records:
    table_rows = sum(1 for line in record["text"].split("\n") if line.strip().startswith("|"))
    print(f"{record['url']}")
    print(f"  title: {record['title']}")
    print(f"  text length: {len(record['text'])} chars")
    print(f"  table rows found: {table_rows}")

print("\nAll sanity checks passed.")

https://www.mom.gov.sg/employment-practices/salary/paying-salary
  title: Paying salary
  text length: 3940 chars
  table rows found: 5
https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days
  title: Hours of work, overtime and rest day
  text length: 6867 chars
  table rows found: 13
https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/work-permit-conditions
  title: Work Permit conditions
  text length: 3053 chars
  table rows found: 0
https://www.mom.gov.sg/passes-and-permits/work-permit-for-foreign-worker/sector-specific-rules/medical-insurance
  title: Medical insurance requirements for migrant workers
  text length: 2917 chars
  table rows found: 4
https://www.mom.gov.sg/contact-us
  title: Contact us
  text length: 1004 chars
  table rows found: 0

All sanity checks passed.
